In [1]:
%run "../../scripts/01-check_setup.ipynb"

SAP HANA Client for Python: 2.29.26061601
Using the dot-env file /Users/I076835/Repositories/GitHub/hana-ai-ve-kg-codejam/.env
Connected to SAP HANA db version 4.00.000.00.1783512515 (fa/CE2026.14) 
at c5889dd5-e0f6-4930-8408-94d53ca61dbf.hna0.prod-us10.hanacloud.ondemand.com:443 as SANDBOX
Current time on the SAP HANA server: 2026-07-13 22:36:04.523000


In [2]:
import glob
from pathlib import Path
import json
import pandas as pd

html_files = sorted(glob.glob("resources/agenda_*.html"))
# html_files = sorted(Path("resources").glob("agenda_*.html"))
print(f"Found {len(html_files)} file(s): {html_files}")

Found 3 file(s): ['resources/agenda_hanatechcon_2026.html', 'resources/agenda_recap_2026.html', 'resources/agenda_ui5con_2026.html']


In [3]:
rows = []
for file_path in html_files:
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    rows.append({"metadata": json.dumps({"file_name": file_path}), "content_html": html_content})
    print(f"  Loaded {len(html_content):,} chars from {file_path}")

df_event = pd.DataFrame(rows).convert_dtypes()
display(df_event.dtypes)
df_event

  Loaded 21,786 chars from resources/agenda_hanatechcon_2026.html
  Loaded 29,703 chars from resources/agenda_recap_2026.html
  Loaded 24,822 chars from resources/agenda_ui5con_2026.html


metadata        string[python]
content_html    string[python]
dtype: object

,metadata,content_html
0,"{""file_name"": ""resources/agenda_hanatechcon_20...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n<me..."
1,"{""file_name"": ""resources/agenda_recap_2026.html""}","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n<me..."
2,"{""file_name"": ""resources/agenda_ui5con_2026.ht...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n<me..."


In [4]:
from IPython.display import HTML

# Read HTML and display for metadata's {"file_name": "resources/agenda_recap_2026.html"}
html_output = f'<base href="resources/images">' + df_event['content_html'].iloc[0]
# content = str(
#     df_event.loc[
#         df_event['metadata'].apply(lambda m: json.loads(m)['file_name'] == 'resources/agenda_recap_2026.html'),
#         'content_html'
#     ].squeeze()
# )
# html_output = content.replace('<head>', '<head><base href="resources/">', 1)

display(HTML(html_output))


In [5]:
import importlib
import config_events
importlib.reload(config_events)
from config_events import HANA_TABLE_NAME, HANA_SCHEMA_NAME

In [6]:
from hana_ml.dataframe import create_dataframe_from_pandas

hdf_event_bronze = create_dataframe_from_pandas(
    connection_context=myconn,
    pandas_df=df_event,
    table_name=HANA_TABLE_NAME,
    schema=HANA_SCHEMA_NAME,
    force=True,
    object_type_as_bin=True,
    table_structure={"metadata": "NVARCHAR(5000)", "content_html": "NCLOB"}
)

100%|██████████| 1/1 [00:00<00:00,  3.09it/s]


In [7]:
pd.set_option("max_colwidth", 256)
hdf_event_bronze.head(1).collect().T

,0
metadata,"{""file_name"": ""resources/agenda_hanatechcon_2026.html""}"
content_html,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n<meta charset=""UTF-8"">\n<title>HTC 2026 Agenda</title>\n</head>\n<body>\n<h1>From Bold Promises to Operational Reality – A Community-Driven Journey</h1>\n<p>9:00–9:40</p>\n<p>Audimax</p>\n<p>SAP HANA: From Bol..."
